In [6]:
import pandas as pd
import numpy as np

from extraction.texts import model_loader, text_encoding

In [11]:
df = pd.read_csv("data/texts/email_spam.csv")
df = pd.concat([
    df[df["label"] == 0][:10000],
    df[df["label"] == 1][:40000]
], axis=0)
texts = df["text"].to_list()

In [12]:
model, model_code = model_loader('mps', model_code='e5')
text_vectors = text_encoding(texts, model, model_code)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [13]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [14]:
x, y = text_vectors, df["label"].to_numpy()
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    stratify=y,
                                                    test_size=0.2)

In [15]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.35,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9768 (When flagged positive, accuracy is 97.68%)
Custom Recall Score:    0.9861 (Captured 98.61% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9773 (When flagged positive, accuracy is 97.73%)
Custom Recall Score:    0.9867 (Captured 98.67% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9759 (When flagged positive, accuracy is 97.59%)
Custom Recall Score:    0.9886 (Captured 98.86% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9759 (When flagged positive, accuracy is 97.59%)
Custom Recall Score:    0.9861 (Captured 98.61% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9787 (When flagged positive, accuracy is 97.87%)
Custom R

In [16]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.35

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

           0       0.94      0.91      0.93      2000
           1       0.98      0.99      0.98      8000

    accuracy                           0.97     10000
   macro avg       0.96      0.95      0.95     10000
weighted avg       0.97      0.97      0.97     10000

[[1819  181]
 [ 112 7888]]
